### Document Structure

In [2]:
from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content= "this is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Dr. Satarupa Biswas",
        "date_created":"2026-08-06"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Dr. Satarupa Biswas', 'date_created': '2026-08-06'}, page_content='this is the main text content I am using to create RAG')

## Create a simple txt  file

In [10]:
import  os
os.makedirs("../data/text_files",exist_ok=True)

In [12]:
sample_texts={
    "../data/text_files/RAG_traditional.txt":"""RAG Introduction
    RAG stands for Retrieval Augmented Generation. It is the process of optimizing the output of an LLM,
    so it references an authoritative knowledge base outside of its training data source before generating a response.
    
    LLMs are trained on vast volumes of data and use billions of parameters to generate original output for  tasks like question answering, translating, and completing sentences.
    
    RAG extends the already powerful capabilities of LLMs to specific domains or an organization's internal knowledge base,
    all without the need to retrain the model.
    it is cost-effective approach to improve LLM output so it remains relevant, accurate, and useful in various context.""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics
    Machine learning is a subset of artificial intelligence that enables systems to learn and improve
    from experience without being explicitly programmed. It focused on developing computer programs
    that can access data and use it to learn for themselves."""
}

for filepath,content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


### How to read these files or some sentences using Text Loader

In [14]:
from langchain_community.document_loaders import TextLoader

### To read the text

In [17]:
loader=TextLoader("../data/text_files/RAG_traditional.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/RAG_traditional.txt'}, page_content="RAG Introduction\n    RAG stands for Retrieval Augmented Generation. It is the process of optimizing the output of an LLM,\n    so it references an authoritative knowledge base outside of its training data source before generating a response.\n\n    LLMs are trained on vast volumes of data and use billions of parameters to generate original output for  tasks like question answering, translating, and completing sentences.\n\n    RAG extends the already powerful capabilities of LLMs to specific domains or an organization's internal knowledge base,\n    all without the need to retrain the model.\n    it is cost-effective approach to improve LLM output so it remains relevant, accurate, and useful in various context.")]


### To read using directory loader, if we have all the files in the directory

In [30]:
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",  ## Pattern to match files
    loader_cls= TextLoader,  ## loader class to use. If we give PDF then we need to mention Pdf Loader. Etc etc
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False

)

In [31]:
documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n    Machine learning is a subset of artificial intelligence that enables systems to learn and improve\n    from experience without being explicitly programmed. It focused on developing computer programs\n    that can access data and use it to learn for themselves.'),
 Document(metadata={'source': '..\\data\\text_files\\RAG_traditional.txt'}, page_content="RAG Introduction\n    RAG stands for Retrieval Augmented Generation. It is the process of optimizing the output of an LLM,\n    so it references an authoritative knowledge base outside of its training data source before generating a response.\n\n    LLMs are trained on vast volumes of data and use billions of parameters to generate original output for  tasks like question answering, translating, and completing sentences.\n\n    RAG extends the already powerful capabilities of LLMs to specific domains or an organization's

### To read the PDF files

In [36]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",  ## Pattern to match files
    loader_cls= PyMuPDFLoader,  ## loader class to use. If we give PDF then we need to mention Pdf Loader. Etc etc
    show_progress=False

)
pdf_documents=dir_loader.load()
pdf_documents


[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-28T14:54:59+05:30', 'source': '..\\data\\pdf\\CNN_projects.pdf', 'file_path': '..\\data\\pdf\\CNN_projects.pdf', 'total_pages': 4, 'format': 'PDF 1.7', 'title': '', 'author': 'Deepratick Biswas', 'subject': '', 'keywords': '', 'moddate': '2026-04-28T14:54:59+05:30', 'trapped': '', 'modDate': "D:20260428145459+05'30'", 'creationDate': "D:20260428145459+05'30'", 'page': 0}, page_content="Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different

In [37]:
type(pdf_documents[0])

langchain_core.documents.base.Document

RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [14]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: CNN_projects.pdf
  ✓ Loaded 4 pages

Processing: gis mega event book.pdf
  ✓ Loaded 252 pages

Total documents loaded: 256


In [15]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-28T14:54:59+05:30', 'author': 'Deepratick Biswas', 'moddate': '2026-04-28T14:54:59+05:30', 'source': '..\\data\\pdf\\CNN_projects.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'CNN_projects.pdf', 'file_type': 'pdf'}, page_content="Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different labs use different dyes (some are more purple, some \nmore pink). I added a ‘color normalization’ step so that the model \ndoesn’t get

In [16]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [17]:
chunks=split_documents(all_pdf_documents)
chunks

Split 256 documents into 12 chunks

Example chunk:
Content: Freelance Projects 
1. Automated Blood Cell Counting 
- What: I automated the task of counting different types of blood cells 
under a microscope (designed for a pathology). 
- Why: A lab-technician h...
Metadata: {'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-28T14:54:59+05:30', 'author': 'Deepratick Biswas', 'moddate': '2026-04-28T14:54:59+05:30', 'source': '..\\data\\pdf\\CNN_projects.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'CNN_projects.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-28T14:54:59+05:30', 'author': 'Deepratick Biswas', 'moddate': '2026-04-28T14:54:59+05:30', 'source': '..\\data\\pdf\\CNN_projects.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'CNN_projects.pdf', 'file_type': 'pdf'}, page_content='Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different labs use different dyes (some are more purple, some \nmore pink). I added a ‘color normalization’ step so that the model \ndoesn’t get

### Embedding and Vectorstore Db

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity 

e:\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Initialize the embedding manager
        
        Args: Model_name: HugginFace model name for sentence embedding"""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a lost of texts.
        Args: texts: List of text strings to embedd
        Returns: numpy array of embeddings with shape (len(texts), embedding_dim)"""
    
        if not self.model:
            raise ValueError("Model not loaded")
    
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings



## Initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager



Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5419.75it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [29]:
import os

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the ector store.
           Args: Collection_name: Name of the ChromaDB collection
           persist_directory: Directory to persist the vector store"""
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistant ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Colection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
           Args: documents: List of LangChain documents
           embeddings: Corresponding embeddings for the documents"""
        
        if len(documents) != len(embeddings):
            raise valueError("Number of document must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document cotent
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

            #Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents to collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Colection: pdf_documents
Existing documents in collection: 0


In [26]:
chunks

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-28T14:54:59+05:30', 'author': 'Deepratick Biswas', 'moddate': '2026-04-28T14:54:59+05:30', 'source': '..\\data\\pdf\\CNN_projects.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'CNN_projects.pdf', 'file_type': 'pdf'}, page_content='Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different labs use different dyes (some are more purple, some \nmore pink). I added a ‘color normalization’ step so that the model \ndoesn’t get

In [27]:
##Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

['Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different labs use different dyes (some are more purple, some \nmore pink). I added a ‘color normalization’ step so that the model \ndoesn’t get confused by different stains. This way, my CNN models \nidentify cells (even overlapping cell) easily. \n- Dataset: The pathology dataset, has thousands of images of blood \nsmears with different types of cells labeled. \n- Output: A full report showing the count and percentage of each cell \ntype. \n \n2. Mapping Tumors in Tissue Scans',
 "smears with different ty

In [30]:
## Generat the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##Store in thr vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]


Generated embeddings with shape: (12, 384)
Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents to collection: 1
Successfully added 12 documents to vector store
Total documents to collection: 2
Successfully added 12 documents to vector store
Total documents to collection: 3
Successfully added 12 documents to vector store
Total documents to collection: 4
Successfully added 12 documents to vector store
Total documents to collection: 5
Successfully added 12 documents to vector store
Total documents to collection: 6
Successfully added 12 documents to vector store
Total documents to collection: 7
Successfully added 12 documents to vector store
Total documents to collection: 8
Successfully added 12 documents to vector store
Total documents to collection: 9
Successfully added 12 documents to vector store
Total documents to collection: 10
Successfully added 12 documents to vector store
Total documents to collection: 11
Successfully added 12 doc

### Retriever Pipeline From VectorStore

In [33]:
class RAGRetriever:
    """Handles query-based retrieval rom the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever
           Args: vector_store: vector store containing document embeddings
           embedding_manager: Manager for generating query embeddings"""
        
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def  retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)




In [34]:
rag_retriever

In [35]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [39]:
rag_retriever.retrieve("What is automated blood cell counting")

Retrieving documents for query: 'What is automated blood cell counting'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_ddd8723d_0',
  'content': 'Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different labs use different dyes (some are more purple, some \nmore pink). I added a ‘color normalization’ step so that the model \ndoesn’t get confused by different stains. This way, my CNN models \nidentify cells (even overlapping cell) easily. \n- Dataset: The pathology dataset, has thousands of images of blood \nsmears with different types of cells labeled. \n- Output: A full report showing the count and percentage of each cell \ntype. \n \n2. Mapping Tumors in Tis

In [40]:
rag_retriever.retrieve("What is gid mega events?")

Retrieving documents for query: 'What is gid mega events?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### RAG Pipeline- VectorDB To LLM Output Generation

In [47]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

None


In [44]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [48]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"





In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = ChatGroq(
        api_key="YOUR_GROQ_API_KEY",
        model_name="llama3-8b-8192" 
    )
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Groq LLM initialized successfully!


In [51]:
rag_retriever.retrieve("What is automated blood cell counting")

Retrieving documents for query: 'What is automated blood cell counting'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_ddd8723d_0',
  'content': 'Freelance Projects \n1. Automated Blood Cell Counting \n- What: I automated the task of counting different types of blood cells \nunder a microscope (designed for a pathology). \n- Why: A lab-technician has to count hundreds of cells one by one. This \nincludes human errors and mistakes. \n- How: I have used CNN for object detection task. The model scans the \nimage, find every cell, and puts a little box around it to identify if it’s \na red cell, white cell, or platelet. \nAlso, different labs use different dyes (some are more purple, some \nmore pink). I added a ‘color normalization’ step so that the model \ndoesn’t get confused by different stains. This way, my CNN models \nidentify cells (even overlapping cell) easily. \n- Dataset: The pathology dataset, has thousands of images of blood \nsmears with different types of cells labeled. \n- Output: A full report showing the count and percentage of each cell \ntype. \n \n2. Mapping Tumors in Tis

### Integration Vectordb Context pipeline With LLM output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


llm=ChatGroq(groq_api_key="YOUR_GROQ_API_KEY",model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [62]:
answer=rag_simple("how to code automated blood cell counting?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'how to code automated blood cell counting?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


To code automated blood cell counting using CNN for object detection, you can follow these steps:

1. **Data Preprocessing**:
   - Load the pathology dataset with thousands of images of blood smears.
   - Apply color normalization to account for different stains.

2. **Model Selection and Training**:
   - Choose a CNN architecture (e.g., YOLO, SSD, Faster R-CNN) for object detection.
   - Train the model on the dataset to detect and classify blood cells.

3. **Object Detection and Classification**:
   - Use the trained model to scan the input image and detect individual cells.
   - Apply a bounding box around each cell and classify it as a red cell, white cell, or platelet.

4. **Post-processing and Reporting**:
   - Count and calculate the percentage of each cell type.
   - Generate a full report showing the count and percentage of each cell type.

Here's a simplified example using Python and the Keras library:

```python
# Import necessary libraries
from keras.models import Sequentia

### Enhanced RAG Pipeline Features

In [64]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output




In [65]:
# Example usage:
result = rag_advanced("pathology dataset", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'pathology dataset'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.78it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: The pathology dataset contains Whole Slide Images (WSI) of lymph nodes from breast cancer patients.
Sources: [{'source': 'CNN_projects.pdf', 'page': 0, 'score': 0.21362507343292236, 'preview': 'and 1 (cancer). By grouping all the high score pixels together, the \nmodel creates a Binary Mask (a black-and-white shape) that perfectly \noutlines the tumor. \n- Dataset: The pathology data, containing WSI of lymph nodes from \nbreast cancer patients. \n- Output: A color-coded map (a heatmap) that ove...'}]
Confidence: 0.21362507343292236
Context Preview: and 1 (cancer). By grouping all the high score pixels together, the 
model creates a Binary Mask (a black-and-white shape) that perfectly 
outlines the tumor. 
- Dataset: The pathology data, containing WSI of lymph nodes from 
breast cancer patients. 
- Output: A color-coded map (a heatmap) that ove


In [66]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [67]:
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("how to map tumors", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'how to map tumors'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
and 1 (cancer). By grouping all the high score pixels together, the 
model creates a Binary Mask (a black-and-white shape) that perfectly 
outlines the tumor. 
- Dataset: The pathology data, containing WSI of lymph nodes from 
breast cancer patients. 


- Output: A color-coded map (a heatmap) that overlays directly on the 
doctor’s screen to show the tumor boundaries.

Question: how to map tumors

Answer:

Final Answer: To map tumors, the model creates a Binary Mask by grouping high-score pixels together, which outlines the tumor perfectly. This Binary Mask is then used to create a color-coded map (heatmap) that overlays directly on the doctor's screen to show the tumor boundaries.

Citations:
[1] CNN_projects.pdf (page 0)
Summary: The model creates a Binary Mask by grouping high-score pixels together to outline the tumor boundaries. This Binary Mask is then used to generate a color-coded heatmap that overlays on the doctor's screen to display the tumor boundaries.
History: {'question': 'how to map tumors', 'answer': "To map tumors, the model creates a Binary Mask by grouping high-score pixels together, which outlines the tumor perfectly. This Binary Mask is then used to create a color-coded map (heatmap) that overlays directly on th